In [3]:
import httpx
from dotenv import load_dotenv
load_dotenv()
from anthropic import Anthropic
client = Anthropic()
model = "claude-sonnet-4-0"

http_client = httpx.Client(verify=False)
client = Anthropic(http_client=http_client)

# Define your local functions (the actual logic)
def get_weather(location):
    # In a real app, you'd call a weather API here
    if "new york" in location.lower():
        return "72°F and Sunny"
    return "65°F and Cloudy"

def calculate_factorial(n):
    import math
    return str(math.factorial(n))

# Define the tool schemas
tools = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a specific location.",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "The city and state, e.g. San Francisco, CA"}
            },
            "required": ["location"]
        }
    },
    {
        "name": "calculate_factorial",
        "description": "Calculate the factorial of a number.",
        "input_schema": {
            "type": "object",
            "properties": {
                "n": {"type": "integer", "description": "The number to calculate the factorial of"}
            },
            "required": ["n"]
        }
    }
]

# %%
# Send the request to Claude
messages = [{"role": "user", "content": "What is the weather in New York and what is the factorial of 6?"}]

response = client.messages.create(
    model=model,
    max_tokens=1024,
    tools=tools,
    messages=messages
)

response

# %%
# Handle the tool calls
# Claude might want to use multiple tools at once
if response.stop_reason == "tool_use":
    # Add Claude's request to the message history
    messages.append({"role": "assistant", "content": response.content})
    
    # Process each tool use block
    for block in response.content:
        if block.type == "tool_use":
            tool_name = block.name
            tool_input = block.input
            tool_use_id = block.id
            
            print(f"Claude wants to use: {tool_name} with input {tool_input}")
            
            # Execute the local function
            if tool_name == "get_weather":
                result = get_weather(tool_input["location"])
            elif tool_name == "calculate_factorial":
                result = calculate_factorial(tool_input["n"])
            
            # Add the result to the messages
            messages.append({
                "role": "user",
                "content": [
                    {
                        "type": "tool_result",
                        "tool_use_id": tool_use_id,
                        "content": result
                    }
                ]
            })

# %%
# Get the final response from Claude
final_response = client.messages.create(
        model=model,
        max_tokens=1024,
        tools=tools,
        messages=messages
    )
print("\nFinal Answer from Claude:")
print(final_response.content[0].text)

C:\Users\61093123\AppData\Local\Temp\ipykernel_17136\601461553.py:52: DeprecationWarning: The model 'claude-sonnet-4-0' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client.messages.create(


Claude wants to use: get_weather with input {'location': 'New York, NY'}
Claude wants to use: calculate_factorial with input {'n': 6}


C:\Users\61093123\AppData\Local\Temp\ipykernel_17136\601461553.py:97: DeprecationWarning: The model 'claude-sonnet-4-0' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  final_response = client.messages.create(



Final Answer from Claude:
Here are your results:

**Weather in New York:** 72°F and Sunny

**Factorial of 6:** 720

The factorial of 6 is calculated as 6 × 5 × 4 × 3 × 2 × 1 = 720.
